# 01 Data Audit

Phase 1 focuses on analytical foundations: verify schema quality, inspect missingness, document cleaning logic, and confirm that normalized outputs preserve title coverage.

This notebook intentionally stays narrow. It is an audit and transformation notebook, not the main business analysis deliverable.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/raw/netflix_titles.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.cleaning import build_processed_outputs, read_raw_titles, save_processed_outputs

RAW_FILE_NAME = "netflix_titles.csv"
RAW_PATH = PROJECT_ROOT / "data/raw" / RAW_FILE_NAME
PROCESSED_PATH = PROJECT_ROOT / "data/processed"


In [2]:
raw_df = read_raw_titles(RAW_PATH)
print(f"Using raw file: {RAW_PATH}")
print(f"Raw shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")
display(raw_df.head(3))

Using raw file: /Users/xinyue/Documents/projects/netflix_da/data/raw/netflix_titles.csv
Raw shape: 8,807 rows x 12 columns


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


In [3]:
schema_audit = pd.DataFrame(
    {
        "column": raw_df.columns,
        "dtype": raw_df.dtypes.astype(str).values,
        "non_null_count": raw_df.notna().sum().values,
        "missing_count": raw_df.isna().sum().values,
        "missing_share": raw_df.isna().mean().round(4).values,
    }
).sort_values(["missing_count", "column"], ascending=[False, True]).reset_index(drop=True)

display(schema_audit)

display(raw_df["type"].value_counts(dropna=False).rename_axis("type").reset_index(name="title_count"))
display(raw_df["rating"].fillna("Missing").value_counts(dropna=False).head(12).rename_axis("rating").reset_index(name="title_count"))

,column,dtype,non_null_count,missing_count,missing_share
0,director,object,6173,2634,0.2991
1,country,object,7976,831,0.0944
2,cast,object,7982,825,0.0937
3,date_added,object,8797,10,0.0011
4,rating,object,8803,4,0.0005
5,duration,object,8804,3,0.0003
6,description,object,8807,0,0.0000
7,listed_in,object,8807,0,0.0000
8,release_year,int64,8807,0,0.0000
9,show_id,string,8807,0,0.0000


,type,title_count
0,Movie,6131
1,TV Show,2676


,rating,title_count
0,TV-MA,3207
1,TV-14,2160
2,TV-PG,863
3,R,799
4,PG-13,490
5,TV-Y7,334
6,TV-Y,307
7,PG,287
8,TV-G,220
9,NR,80


In [4]:
date_parse_check = pd.DataFrame(
    {
        "check": [
            "raw_missing_date_added",
            "naive_parse_failures",
            "explicit_parse_failures_after_trim",
        ],
        "value": [
            raw_df["date_added"].isna().sum(),
            pd.to_datetime(raw_df["date_added"], errors="coerce").isna().sum(),
            pd.to_datetime(
                raw_df["date_added"].astype("string").str.strip(),
                format="%B %d, %Y",
                errors="coerce",
            ).isna().sum(),
        ],
    }
)

display(date_parse_check)

,check,value
0,raw_missing_date_added,10
1,naive_parse_failures,98
2,explicit_parse_failures_after_trim,10


## Cleaning Rules Applied in Phase 1

- Preserve all raw title rows.
- Trim whitespace before parsing `date_added` with the explicit format `%B %d, %Y`.
- Keep missing values visible instead of dropping rows globally.
- Normalize `country`, `listed_in`, `cast`, and `director` into bridge tables.
- Preserve raw `rating` and add heuristic portfolio groupings with `rating_system`, `rating_group`, and `is_unrated`.
- Preserve raw `duration` and parse it into `duration_value` plus `duration_unit`.

In [5]:
processed_tables, qa_outputs = build_processed_outputs(raw_df)
save_processed_outputs(processed_tables, qa_outputs, PROCESSED_PATH)
print(f"Saved processed outputs to: {PROCESSED_PATH}")

display(qa_outputs["qa_table_counts"])
display(qa_outputs["qa_parse_checks"])
display(qa_outputs["qa_bridge_coverage"])
display(qa_outputs["qa_titles_missingness"].head(10))

Saved processed outputs to: /Users/xinyue/Documents/projects/netflix_da/data/processed


,table_name,row_count,distinct_show_id
0,raw_titles,8807,8807
1,titles,8807,8807
2,title_country,10012,7976
3,title_genre,19323,8807
4,title_cast,64124,7982
5,title_director,6977,6173


,check_name,value
0,duplicate_show_id_raw,0
1,duplicate_show_id_titles,0
2,raw_date_added_missing,10
3,date_added_parse_failures_non_missing,0
4,duration_parse_failures,3
5,unrated_or_unknown_titles,87


,source_column,bridge_table,titles_with_values_raw,titles_with_values_bridge,bridge_rows,avg_values_per_title
0,country,title_country,7976,7976,10012,1.26
1,listed_in,title_genre,8807,8807,19323,2.19
2,cast,title_cast,7982,7982,64124,8.03
3,director,title_director,6173,6173,6977,1.13


,column,missing_count,missing_share
0,date_added,10,0.0011
1,date_added_month,10,0.0011
2,date_added_year,10,0.0011
3,rating,4,0.0005
4,duration,3,0.0003
5,duration_unit,3,0.0003
6,duration_value,3,0.0003
7,description,0,0.0000
8,is_unrated,0,0.0000
9,rating_group,0,0.0000


In [6]:
processed_inventory = pd.DataFrame(
    {"file_name": sorted(path.name for path in PROCESSED_PATH.glob("*.csv"))}
)
display(processed_inventory)
print(f"Processed inventory refreshed from: {PROCESSED_PATH}")

,file_name
0,qa_bridge_coverage.csv
1,qa_parse_checks.csv
2,qa_table_counts.csv
3,qa_titles_missingness.csv
4,title_cast.csv
5,title_country.csv
6,title_director.csv
7,title_genre.csv
8,titles.csv


Processed inventory refreshed from: /Users/xinyue/Documents/projects/netflix_da/data/processed


## Phase 1 Outcome

At the end of this notebook, the project has an analysis-ready base table, normalized entity bridges, and QA artifacts that are written back to `data/processed/`. Later notebooks depend on these refreshed outputs.